In [1]:
!pip install selenium requests pandas Pillow beautifulsoup4 lxml

In [2]:
import time, re, requests, pandas as pd, os
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import shutil

In [3]:
options = Options()
options.binary_location = "/usr/bin/chromium"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
chromedriver_path = shutil.which("chromedriver") or "/usr/lib/chromium/chromedriver"
driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
print("Driver ready ✅")

Driver ready ✅


In [4]:
driver.get("https://www.houzz.com/photos/bedroom-ideas-and-designs-phbr0-bp~t_715")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
time.sleep(4)
print("Page loaded ✅  Title:", driver.title)

Page loaded ✅  Title: 75 Bedroom Ideas You'll Love - April, 2026 | Houzz


In [5]:
def collect_page(driver, room_type):
    for scroll in range(0, 6000, 600):
        driver.execute_script(f"window.scrollTo(0, {scroll});")
        time.sleep(0.3)
    time.sleep(2)

    soup    = BeautifulSoup(driver.page_source, "lxml")
    results = []
    seen    = set()

    SKIP = ("save","view","add to","sign in","join","explore","browse",
            "see more","load more","similar","get","find","discover",
            "shop","click","learn","read")

    ROOM_KEYS = {"floor","wall","ceiling","kitchen","bedroom","bathroom",
                 "living","dining","office","cabinet","countertop","island",
                 "design","photo","example","style","space","room","layout",
                 "storage","renovation","remodel","transitional","modern",
                 "traditional","contemporary","farmhouse","light","window"}

    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if not src.startswith("http"): continue

        m = re.search(r"-w(\d+)-", src)
        if not m or int(m.group(1)) < 300: continue
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue

        caption = None
        node = img

        for _ in range(6):
            node = node.parent
            if node is None: break

            for child in node.find_all(["p","span","div","a"], recursive=False):
                text = child.get_text(separator=" ", strip=True)
                words = text.split()

                if len(words) < 10: continue
                if text.lower().startswith(SKIP): continue

                lower_count = sum(1 for w in words if w.islower())
                if lower_count < 5: continue

                if not set(text.lower().split()) & ROOM_KEYS: continue

                caption = text
                break

            if caption: break

        if not caption: continue

        title = (img.get("alt") or "").strip() or " ".join(caption.split()[:6])
        seen.add(src)
        results.append({"title":title,"description":caption,
                        "image_url":src,"room_type":room_type})
    return results

In [6]:
page_data = collect_page(driver, "bedroom")
print(f"Page 1: {len(page_data)} good captions")
for item in page_data[:5]:
    nw = len(item["description"].split())
    print(f"  [{nw} words] {item['description'][:85]}")

Page 1: 39 good captions
  [15 words] Mid-sized trendy master gray floor and vinyl floor bedroom photo in Phoenix with gray
  [17 words] Inspiration for a large 1960s master carpeted bedroom remodel in Miami with yellow wa
  [98 words] This industrial condo in Roger's Park got a complete overhaul. The layout remained wh
  [109 words] Bedroom Ideas & Designs Whether you want inspiration for planning a bedroom renovatio
  [109 words] Bedroom Ideas & Designs Whether you want inspiration for planning a bedroom renovatio


In [7]:
URLS  = ['https://www.houzz.com/photos/bedroom-ideas-and-designs-phbr0-bp~t_715']
PAGES = 400
TARGET = 4000
all_data = []
all_seen = set()
print("Room:", "bedroom", "| Target:", TARGET)

for ui, base_url in enumerate(URLS):
    style = base_url.split("~s_")[-1] if "~s_" in base_url else "main"
    print(f"\n  URL {ui+1}/{len(URLS)} — {style}")
    empty = 0
    for page in range(1, PAGES+1):
        if len(all_data) >= TARGET:
            print("  ✅ Target reached"); break
        driver.get(f"{base_url}?pg={page}")
        print(f"    Page {page:03d}", end=" → ")
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
        except:
            print("TIMEOUT"); empty += 1
            if empty >= 3: print("  next URL"); break
            continue
        new = [i for i in collect_page(driver, "bedroom") if i["image_url"] not in all_seen]
        for i in new: all_seen.add(i["image_url"])
        all_data.extend(new)
        print(f"{len(new):3d} new | total: {len(all_data)}")
        empty = 0 if new else empty + 1
        if empty >= 5: print("  No new — next URL"); break
        time.sleep(1.5)
    if len(all_data) >= TARGET: break

print(f"\n✅ Done. Total: {len(all_data)} images")

Room: bedroom | Target: 4000

  URL 1/1 — main
    Page 001 →  20 new | total: 20
    Page 002 →  20 new | total: 40
    Page 003 →  20 new | total: 60
    Page 004 →  20 new | total: 80
    Page 005 →  20 new | total: 100
    Page 006 →  20 new | total: 120
    Page 007 →  20 new | total: 140
    Page 008 →  20 new | total: 160
    Page 009 →  20 new | total: 180
    Page 010 →  20 new | total: 200
    Page 011 →  20 new | total: 220
    Page 012 →  20 new | total: 240
    Page 013 →  20 new | total: 260
    Page 014 →  20 new | total: 280
    Page 015 →  20 new | total: 300
    Page 016 →  20 new | total: 320
    Page 017 →  20 new | total: 340
    Page 018 →  20 new | total: 360
    Page 019 →  20 new | total: 380
    Page 020 →  20 new | total: 400
    Page 021 →  20 new | total: 420
    Page 022 →  20 new | total: 440
    Page 023 →  20 new | total: 460
    Page 024 →  20 new | total: 480
    Page 025 →  20 new | total: 500
    Page 026 →  20 new | total: 520
    Page 027 →  20 ne

In [8]:
driver.quit()
print('Driver closed ✅')

Driver closed ✅


In [9]:
df = pd.DataFrame(all_data).drop_duplicates(subset=["image_url"]).reset_index(drop=True)
df["wc"] = df["description"].str.split().str.len()
print(f"Total: {len(df)} | Min words: {df['wc'].min()} | Avg: {df['wc'].mean():.1f} | All≥10: {(df['wc']>=10).all()} ✅")
df.head()

Total: 4009 | Min words: 10 | Avg: 133.6 | All≥10: True ✅


,title,description,image_url,room_type,wc
0,Central Phoenix Master Bedroom,Mid-sized trendy master gray floor and vinyl f...,https://st.hzcdn.com/fimgs/d401eec203fbde1a_51...,bedroom,15
1,Grey Oaks,Inspiration for a large 1960s master carpeted ...,https://st.hzcdn.com/fimgs/23b12295073b2d5e_95...,bedroom,17
2,Industrial Condo Primary Bedroom,This industrial condo in Roger's Park got a co...,https://st.hzcdn.com/fimgs/2811b5af05303a06_73...,bedroom,98
3,Isle of Palms Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/86d12b9c0a7477a7_89...,bedroom,111
4,Manchester By The Sea Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/7de16dd701f1a4c4_99...,bedroom,111


In [10]:
os.makedirs("images/bedroom", exist_ok=True)
H = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}
lp, fail = [], 0
for i, row in df.iterrows():
    fname = "images/bedroom/bedroom_" + str(i).zfill(5) + ".jpg"
    try:
        r = requests.get(row["image_url"], headers=H, timeout=15)
        if r.status_code == 200:
            Image.open(BytesIO(r.content)).convert("RGB").save(fname)
            lp.append(fname)
        else: lp.append(None); fail+=1
    except: lp.append(None); fail+=1
    time.sleep(0.4)
df["local_path"] = lp
print(f"Downloaded: {len(df)-fail} | Failed: {fail}")

Downloaded: 4009 | Failed: 0


In [11]:
df = df.dropna(subset=["local_path"]).drop(columns=["wc"],errors="ignore").reset_index(drop=True)
print(f"Final: {len(df)} images")
df.head()

Final: 4009 images


,title,description,image_url,room_type,local_path
0,Central Phoenix Master Bedroom,Mid-sized trendy master gray floor and vinyl f...,https://st.hzcdn.com/fimgs/d401eec203fbde1a_51...,bedroom,images/bedroom/bedroom_00000.jpg
1,Grey Oaks,Inspiration for a large 1960s master carpeted ...,https://st.hzcdn.com/fimgs/23b12295073b2d5e_95...,bedroom,images/bedroom/bedroom_00001.jpg
2,Industrial Condo Primary Bedroom,This industrial condo in Roger's Park got a co...,https://st.hzcdn.com/fimgs/2811b5af05303a06_73...,bedroom,images/bedroom/bedroom_00002.jpg
3,Isle of Palms Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/86d12b9c0a7477a7_89...,bedroom,images/bedroom/bedroom_00003.jpg
4,Manchester By The Sea Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/7de16dd701f1a4c4_99...,bedroom,images/bedroom/bedroom_00004.jpg


In [12]:
df.to_csv("bedroom_dataset.csv", index=False)
print("Saved → bedroom_dataset.csv ✅")

Saved → bedroom_dataset.csv ✅


In [13]:
pd.read_csv("bedroom_dataset.csv")

,title,description,image_url,room_type,local_path
0,Central Phoenix Master Bedroom,Mid-sized trendy master gray floor and vinyl f...,https://st.hzcdn.com/fimgs/d401eec203fbde1a_51...,bedroom,images/bedroom/bedroom_00000.jpg
1,Grey Oaks,Inspiration for a large 1960s master carpeted ...,https://st.hzcdn.com/fimgs/23b12295073b2d5e_95...,bedroom,images/bedroom/bedroom_00001.jpg
2,Industrial Condo Primary Bedroom,This industrial condo in Roger's Park got a co...,https://st.hzcdn.com/fimgs/2811b5af05303a06_73...,bedroom,images/bedroom/bedroom_00002.jpg
3,Isle of Palms Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/86d12b9c0a7477a7_89...,bedroom,images/bedroom/bedroom_00003.jpg
4,Manchester By The Sea Home,Bedroom Ideas & Designs Whether you want inspi...,https://st.hzcdn.com/fimgs/7de16dd701f1a4c4_99...,bedroom,images/bedroom/bedroom_00004.jpg
...,...,...,...,...,...
4004,Navy & Gold Bedroom,This guest bedroom was previously done in shad...,https://st.hzcdn.com/fimgs/d921a4fc07aa3b71_60...,bedroom,images/bedroom/bedroom_04004.jpg
4005,Holly Springs Drive,Example of a mid-sized transitional master med...,https://st.hzcdn.com/fimgs/21917f5a0d9b70a8_72...,bedroom,images/bedroom/bedroom_04005.jpg
4006,WEST VILLAGE TOWNHOUSE,All Filters Style Contemporary Modern Traditio...,https://st.hzcdn.com/fimgs/baf1fc3d08f4e644_44...,bedroom,images/bedroom/bedroom_04006.jpg
4007,Lennar at Landmark - House #2,Bedroom\nResidential home \nLocation: Landmark...,https://st.hzcdn.com/fimgs/71c1e0520a5bdc6c_02...,bedroom,images/bedroom/bedroom_04007.jpg
